# Banking Injection Analysis — Causal Understanding Pipeline

**Goal**: understand *why* prompt injection payloads succeed or fail in the banking
environment, and use that understanding to guide token substitutions.

**Structure**:
1. **Discriminative** — does the attention signature generalise across unseen
   injections (cross-injection LOO) and unseen user tasks (cross-user-task LOO)?
2. **Contrastive causal** — what changes in attention when the same context produces
   a different outcome due to a token substitution?
3. **Position-level** — which payload positions are structurally critical vs robust?
4. **Cross-reference** — do the discriminative and causal analyses agree on which
   heads matter? Agreement = the classifier found the causal mechanism, not a correlate.

Data: `profiling_logs_v10` (banking, N∈{1,2,4}, 10,755 captures).
Extended analysis with N∈{1,2,4,8,16,32} + MLP outputs + logit lens: `profiling_logs_v12`.


In [ ]:
import sys, sqlite3, json, dill, pickle, torch, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

plt.rcParams.update({
    'figure.dpi': 120, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
})

N_LAYERS          = 24
N_HEADS           = 64
LOCAL_LAYERS      = list(range(0, N_LAYERS, 2))
GLOBAL_LAYERS     = list(range(1, N_LAYERS, 2))
SINK_POSITIONS    = {0}
N_REASONING_STEPS = 10

SPAN_GROUPS = {
    'attack_payload':   {'attack_payload'},
    'attack_prefix':    {'attack_prefix'},
    'attack_suffix':    {'attack_suffix'},
    'user_instruction': {'user_instruction'},
    'tool_env_data':    {'tool_env_data'},
    'dev_instructions': {'developer_instructions'},
    'dev_tools':        {'developer_tools'},
    'system_meta':      {'system_meta'},
    'frame_boundary':   {'frame_boundary'},
    'frame_message':    {'frame_message'},
    'frame_role':       {'frame_role'},
    'frame_channel':    {'frame_channel'},
    'frame_constrain':  {'frame_constrain'},
    'frame_chan_name':  {'frame_channel_name'},
    'frame_cons_type':  {'frame_constrain_type'},
    'frame_metadata':   {'frame_metadata'},
}
GROUP_NAMES = list(SPAN_GROUPS.keys()) + ['sink']
N_GROUPS    = len(GROUP_NAMES)   # 17

GROUP_COLOR = {
    'attack_payload': '#c0392b', 'attack_prefix': '#e74c3c', 'attack_suffix': '#ff7675',
    'user_instruction': '#3498db', 'tool_env_data': '#f39c12',
    'dev_instructions': '#8e44ad', 'dev_tools': '#9b59b6',
    'system_meta': '#7f8c8d',
    'frame_boundary': '#bdc3c7', 'frame_message': '#b2bec3', 'frame_role': '#adb5bd',
    'frame_channel': '#a8a8a8', 'frame_constrain': '#c0c0c0',
    'frame_chan_name': '#cacaca', 'frame_cons_type': '#d4d4d4', 'frame_metadata': '#dedede',
    'sink': '#1abc9c',
}

LOG_DIR     = Path('profiling_logs_v10')   # update if needed
TENSOR_BASE = Path('.')
CACHE_FILE  = LOG_DIR / 'spark_cache.pkl'

print(f'Groups: {N_GROUPS}   Log dir: {LOG_DIR}')


In [ ]:
conn = sqlite3.connect(LOG_DIR / 'experiment_logs.db')
cur  = conn.cursor()

cur.execute("SELECT metadata_json FROM logs WHERE event='profiling_run'")
runs_meta = {json.loads(r[0])['run_id']: json.loads(r[0]) for r in cur.fetchall()}

cur.execute("SELECT metadata_json, object_data FROM logs WHERE event='profiling_capture'")
cap_by_run = {}
for meta_json, obj_data in cur.fetchall():
    rid = json.loads(meta_json)['run_id']
    cap_by_run[rid] = dill.loads(obj_data)

conn.close()
print(f'Runs: {len(runs_meta)}   Captures: {len(cap_by_run)}')

def _payload_positions(cap_info):
    payload_tags = {'attack_payload', 'attack_prefix', 'attack_suffix'}
    positions = set()
    for span in cap_info.get('spans', []):
        if span.get('tag') in payload_tags:
            positions.update(range(span['start'], span['end']))
    return positions

# Index baseline captures
baseline_cap_by_ctx = {}
for rid, m in runs_meta.items():
    if m.get('source') == 'baseline' and rid in cap_by_run:
        ctx = (m['suite'], m['injection_task_id'], m['user_task_id'])
        baseline_cap_by_ctx[ctx] = cap_by_run[rid]

print(f'Baseline contexts indexed: {len(baseline_cap_by_ctx)}')

# Compute true_cumulative_N and record which positions were flipped
n_exact = 0
for rid, m in runs_meta.items():
    ctx     = (m.get('suite'), m.get('injection_task_id'), m.get('user_task_id'))
    bl      = baseline_cap_by_ctx.get(ctx)
    run_cap = cap_by_run.get(rid)

    if bl and run_cap:
        bl_ids  = bl.get('token_ids', [])
        run_ids_tok = run_cap.get('token_ids', [])
        if bl_ids and run_ids_tok and len(bl_ids) == len(run_ids_tok):
            payload_pos = _payload_positions(bl)
            flipped = [p for p in payload_pos if bl_ids[p] != run_ids_tok[p]]
            m['true_cumulative_N']  = len(flipped)
            m['flipped_positions']  = flipped
            n_exact += 1
            continue

    own_N = int(m.get('perturbation_N') or 0)
    if m.get('source') == 'failure_tree':
        parent   = runs_meta.get(m.get('parent_run_id', ''))
        parent_N = int(parent.get('perturbation_N') or 0) if parent else 0
        m['true_cumulative_N'] = own_N + parent_N
    else:
        m['true_cumulative_N'] = own_N
    m['flipped_positions'] = []

print(f'true_cumulative_N: {n_exact} exact (token diff), {len(runs_meta)-n_exact} additive fallback')

# Banking flip runs with captures
flip_runs    = [m for m in runs_meta.values()
                if m.get('perturbation_type') == 'flip' and m['run_id'] in cap_by_run]
banking_flips = [m for m in flip_runs if m.get('suite') == 'banking']
print(f'\nBanking flip runs with captures: {len(banking_flips)}')
print(f'  success: {sum(1 for r in banking_flips if r.get("success"))}')
print(f'  failure: {sum(1 for r in banking_flips if not r.get("success"))}')


In [ ]:
# Groups = (injection_task_id, user_task_id, true_cumulative_N)
meta_by_rid = {m['run_id']: m for m in runs_meta.values()}

raw_groups = defaultdict(list)
for m in banking_flips:
    key = (m['injection_task_id'], m['user_task_id'], m['true_cumulative_N'])
    raw_groups[key].append(m['run_id'])

# Keep only groups with >=3 success AND >=3 failure (enough for a mean estimate)
MIN_PER_CLASS = 3
mixed_groups = {}
for key, rids in raw_groups.items():
    n_s = sum(1 for r in rids if meta_by_rid[r].get('success'))
    n_f = sum(1 for r in rids if not meta_by_rid[r].get('success'))
    if n_s >= MIN_PER_CLASS and n_f >= MIN_PER_CLASS:
        mixed_groups[key] = {'run_ids': rids, 'n_success': n_s, 'n_failure': n_f}

print(f'Total groups (all N): {len(raw_groups)}')
print(f'Mixed groups (>={MIN_PER_CLASS} success + >={MIN_PER_CLASS} failure): {len(mixed_groups)}')
print()

from collections import Counter
by_inj = Counter(k[0] for k in mixed_groups)
for inj, cnt in sorted(by_inj.items()):
    inj_s = inj.replace('injection_task_', 'it')
    runs_in_inj = sum(len(mixed_groups[k]['run_ids']) for k in mixed_groups if k[0] == inj)
    Nvals = sorted(set(k[2] for k in mixed_groups if k[0] == inj))
    print(f'  {inj_s}: {cnt} groups, {runs_in_inj} runs, N ∈ {Nvals}')


---
## Part 1 — Discriminative analysis: does the attention signature generalise?

Before asking *what* the causal signature is, establish *whether* it generalises
across unseen injections and unseen user tasks. Two evaluations:

1. **Cross-injection LOO**: train on N−1 banking injections, test on held-out 1.
   Attacker knows the environment (banking) but not the specific injection payload.
2. **Cross-user-task LOO**: within a single injection, train on one user-task context,
   test on another. Attacker knows the injection but not the user's goal.

Hyperparameters selected on a val injection (clean nested split — no leakage from test).
Best feature variant and head-level importances feed directly into Parts 2 and 3.


In [ ]:
import pickle, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn.model_selection import StratifiedKFold
from itertools import product as _itp

PLOTS_DIR  = LOG_DIR / 'plots'
MODELS_DIR = LOG_DIR / 'models'
PLOTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

# ── Load cached step-0 / trajectory feature arrays ────────────────────────────
# These were built once by the per_injection analysis pipeline and are reused here.
_s0m  = np.load(LOG_DIR / 'feature_cache_step0_mass.npy',  mmap_mode='r')  # [N, 26112]
_s0e  = np.load(LOG_DIR / 'feature_cache_step0_ent.npy',   mmap_mode='r')
_t5m  = np.load(LOG_DIR / 'feature_cache_traj5_mass.npy',  mmap_mode='r')
_t10m = np.load(LOG_DIR / 'feature_cache_traj10_mass.npy', mmap_mode='r')

with open(LOG_DIR / 'feature_cache_meta.pkl', 'rb') as f:
    _cache_meta = pickle.load(f)
_smeta  = np.load(str(LOG_DIR / 'feature_cache_smeta.npz'))
y_loi   = _smeta['y'].astype(np.int32)
N_CACHE = len(_cache_meta)

_inj_keys  = np.array([m.get('injection_task_id','') for m in _cache_meta])
_ut_keys   = np.array([m.get('user_task_id','')      for m in _cache_meta])
_suite_keys= np.array([m.get('suite','')              for m in _cache_meta])
_ctx_keys  = np.array([f"{m.get('injection_task_id','')}/{m.get('user_task_id','')}"
                        for m in _cache_meta])

banking_injections_loi = sorted(set(_inj_keys[_suite_keys == 'banking']))
print(f'Feature cache: {N_CACHE} runs')
print(f'Banking injections: {banking_injections_loi}')

# ── Feature variants for LOI ─────────────────────────────────────────────────
def _gl(X):   # all-layers [N,26112] → global-only [N,13056]
    return np.asarray(X).reshape(N_CACHE, N_LAYERS, N_HEADS, N_GROUPS)\
                        [:, GLOBAL_LAYERS].reshape(N_CACHE, -1)

LOI_FEATURES = {
    'step0_global':          _gl(_s0m),
    'step0_all':             np.asarray(_s0m),
    'step0_global+ent':      np.hstack([_gl(_s0m), _gl(_s0e)]),
    'traj5_global':          _gl(_t5m),
    'traj10_global':         _gl(_t10m),
}

# Contrastive normalisation: subtract per-context failure mean
# Leverages the large failure pool to remove context-level confounding.
_fail_mask = (y_loi == 0)
for _bname in list(LOI_FEATURES.keys()):
    _X    = LOI_FEATURES[_bname]
    _Xout = _X.copy()
    for _ctx in np.unique(_ctx_keys[_suite_keys == 'banking']):
        _cm = _ctx_keys == _ctx
        _fm = _cm & _fail_mask
        if _fm.sum() < 10: continue
        _Xout[_cm] -= _X[_fm].mean(axis=0)
    LOI_FEATURES[_bname + '_cnorm'] = _Xout

EVAL_FEAT = list(LOI_FEATURES.keys())
print(f'LOI feature variants ({len(EVAL_FEAT)}):')
for n, X in LOI_FEATURES.items():
    print(f'  {n:30s}: {X.shape[1]:>7} features')


In [ ]:
# Grid biased toward strong regularisation — training sets are small
# (2 injections, sometimes <2000 samples on 13-26k features).
HYPERPARAM_GRID = {
    'LogReg': [{'C': c} for c in [0.0001, 0.001, 0.01, 0.1]],
    'RF':     [{'min_samples_leaf': msl} for msl in [5, 10, 20, 50]],
}

def make_loi_models():
    return {
        'LogReg': LogisticRegression(C=0.01, class_weight='balanced',
                                      max_iter=500, tol=1e-3,
                                      solver='saga', random_state=42),
        'RF':     RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                          max_features='sqrt', min_samples_leaf=10,
                                          n_jobs=-1, random_state=42),
    }
print('Hyperparam grid defined.')


In [ ]:
# ── Cross-injection LOO ────────────────────────────────────────────────────────
# Rotation: test=itX  val=it(X+1 mod 4)  train=remaining 2
# Hyperparams selected on val (fit on train, score on val).
# Final model fit on train+val, evaluated on test.

loi_results = []
best_configs_loi = {}   # {test_inj: {feat: {model: best_params}}}

for i, test_inj in enumerate(banking_injections_loi):
    val_inj    = banking_injections_loi[(i+1) % len(banking_injections_loi)]
    train_injs = [inj for inj in banking_injections_loi
                  if inj not in (test_inj, val_inj)]

    test_m  = _inj_keys == test_inj
    val_m   = _inj_keys == val_inj
    train_m = np.isin(_inj_keys, train_injs)
    tv_m    = train_m | val_m

    y_te = y_loi[test_m];  y_va = y_loi[val_m];  y_tr = y_loi[train_m]
    n_s  = int(y_te.sum()); n_f = int((1-y_te).sum())
    if n_s < 5 or n_f < 5:
        print(f'Skip {test_inj}: succ={n_s} fail={n_f}')
        continue

    ts = test_inj.replace('injection_task_','it')
    vs = val_inj.replace('injection_task_','it')
    trs = '+'.join(t.replace('injection_task_','it') for t in train_injs)
    print(f'\nTest={ts}  Val={vs}  Train={trs}')
    print(f'  train n={train_m.sum()} val n={val_m.sum()} test n={test_m.sum()} rate={y_te.mean():.2f}')

    row = {'held_out': test_inj, 'short': ts, 'val': val_inj,
           'n_test': int(test_m.sum()), 'n_succ': n_s, 'n_fail': n_f,
           'rate': float(y_te.mean())}
    best_configs_loi[test_inj] = {}

    for feat_name in EVAL_FEAT:
        X_f = LOI_FEATURES[feat_name]
        best_configs_loi[test_inj][feat_name] = {}

        for model_name, clf_t in make_loi_models().items():
            # val selection
            best_va, best_p = -1, {}
            for params in HYPERPARAM_GRID.get(model_name, [{}]):
                clf_c = clone(clf_t).set_params(**params)
                clf_c.fit(X_f[train_m], y_tr)
                va = roc_auc_score(y_va, clf_c.predict_proba(X_f[val_m])[:,1])
                if va > best_va:
                    best_va, best_p = va, params
            best_configs_loi[test_inj][feat_name][model_name] = best_p

            # final model
            clf_f = clone(clf_t).set_params(**best_p)
            clf_f.fit(X_f[tv_m], y_loi[tv_m])
            te = roc_auc_score(y_te, clf_f.predict_proba(X_f[test_m])[:,1])
            row[f'{feat_name}/{model_name}'] = round(te, 4)
            print(f'  {feat_name:32s} + {model_name:8s}: val={best_va:.4f} test={te:.4f} {best_p}')
            joblib.dump(clf_f, MODELS_DIR / f'loi_{ts}_{feat_name}_{model_name}.pkl')

    loi_results.append(row)

loi_df = pd.DataFrame(loi_results).set_index('held_out') if loi_results else None
if loi_df is not None:
    auroc_cols = [c for c in loi_df.columns if '/' in c]
    print('\nMean LOI AUROC per feature×model:')
    means = loi_df[auroc_cols].mean().sort_values(ascending=False)
    print(means.round(3).to_string())
    BEST_LOI_COL   = means.idxmax()
    BEST_LOI_FEAT, BEST_LOI_MODEL = BEST_LOI_COL.rsplit('/', 1)
    print(f'\nBest: {BEST_LOI_FEAT}  +  {BEST_LOI_MODEL}')
    loi_df.to_csv(LOG_DIR / 'loi_results.csv')


In [ ]:
# ── LOI AUROC heatmap + feature importance ────────────────────────────────────
if loi_df is not None:
    auroc_cols = [c for c in loi_df.columns if '/' in c]
    mat = loi_df[auroc_cols].values.astype(float)

    fig, ax = plt.subplots(figsize=(max(12, len(auroc_cols)*0.9+2), len(loi_df)*1.2+2))
    im = ax.imshow(mat, aspect='auto', cmap='RdYlGn', vmin=0.45, vmax=1.0,
                   interpolation='nearest')
    ax.set_xticks(range(len(auroc_cols)))
    ax.set_xticklabels([c.replace('step0_','s0_').replace('traj','tr').replace('global','gl')
                         .replace('/','\n') for c in auroc_cols],
                        rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(loi_df)))
    ax.set_yticklabels([r.replace('injection_task_','it') for r in loi_df.index])
    for ri in range(len(loi_df)):
        for ci in range(len(auroc_cols)):
            v = mat[ri,ci]
            if not np.isnan(v):
                ax.text(ci, ri, f'{v:.3f}', ha='center', va='center', fontsize=7)
    plt.colorbar(im, ax=ax, shrink=0.6)
    ax.set_title('Cross-injection LOO AUROC\n(test injection × feature+model, clean nested split)',
                 fontsize=10)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'loi_auroc_heatmap.png', bbox_inches='tight')
    plt.show(); plt.close('all')

    # Feature importance: |coef| for LogReg on best feat, summed to layer×head
    loi_imp_by_inj = {}
    for row_dict in loi_results:
        held_out = row_dict['held_out']; ts = row_dict['short']
        fname = f'loi_{ts}_{BEST_LOI_FEAT}_{BEST_LOI_MODEL}.pkl'
        try:
            clf = joblib.load(MODELS_DIR / fname)
        except FileNotFoundError:
            continue
        if hasattr(clf, 'coef_'):
            raw = np.abs(clf.coef_[0])
            n_gl = len(GLOBAL_LAYERS) if 'global' in BEST_LOI_FEAT else N_LAYERS
            imp  = raw[:n_gl*N_HEADS*N_GROUPS].reshape(n_gl, N_HEADS, N_GROUPS)
            loi_imp_by_inj[held_out] = imp.sum(-1)   # [n_layers, H]

    if loi_imp_by_inj:
        # Mean importance across held-out injections
        mean_loi_imp = np.mean(list(loi_imp_by_inj.values()), axis=0)
        layers_used  = GLOBAL_LAYERS if 'global' in BEST_LOI_FEAT else list(range(N_LAYERS))

        fig, ax = plt.subplots(figsize=(14, 3.5))
        im = ax.imshow(mean_loi_imp.T, aspect='auto', cmap='viridis',
                       interpolation='nearest', origin='lower')
        ax.set_xlabel('Layer index'); ax.set_ylabel('Head')
        ax.set_title(f'LOI feature importance (mean |coef|) — {BEST_LOI_FEAT} + {BEST_LOI_MODEL}')
        ax.set_xticks(range(len(layers_used)))
        ax.set_xticklabels([f'{l}\n{"L" if l in LOCAL_LAYERS else "G"}' for l in layers_used], fontsize=6)
        plt.colorbar(im, ax=ax, shrink=0.8)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / 'loi_feature_importance.png', bbox_inches='tight')
        plt.show(); plt.close('all')

        LOI_LH_IMP = mean_loi_imp   # [n_layers, H] — used in cross-reference
        print(f'LOI importance computed for {len(loi_imp_by_inj)} held-out injections')


In [ ]:
# ── Cross-user-task LOO ────────────────────────────────────────────────────────
# For injections with >=2 user-task contexts with captures, hold out one user task
# and train on all other contexts (all injections, other user tasks).
# Tests: does the signature transfer when the payload is fixed but user goal changes?

xut_results = []
multi_ut_injs = {inj: sorted(set(_ut_keys[_inj_keys == inj]))
                  for inj in banking_injections_loi
                  if len(set(_ut_keys[_inj_keys == inj])) >= 2}

if not multi_ut_injs:
    print('No injections with >=2 user tasks in cache — skipping cross-user-task LOO')
else:
    print('Cross-user-task LOO:')
    for inj, uts in multi_ut_injs.items():
        inj_s = inj.replace('injection_task_','it')
        for test_ut in uts:
            test_m  = (_inj_keys == inj) & (_ut_keys == test_ut)
            train_m = ~test_m   # all other contexts (other inj + other ut of same inj)
            y_te = y_loi[test_m]; y_tr = y_loi[train_m]
            if y_te.sum() < 5 or (1-y_te).sum() < 5:
                continue
            X_f = LOI_FEATURES[BEST_LOI_FEAT]
            clf = clone(make_loi_models()[BEST_LOI_MODEL])
            clf.set_params(**best_configs_loi.get(inj, {}).get(BEST_LOI_FEAT, {}).get(BEST_LOI_MODEL, {}))
            clf.fit(X_f[train_m], y_tr)
            auc = roc_auc_score(y_te, clf.predict_proba(X_f[test_m])[:,1])
            ut_s = test_ut.replace('user_task_','ut')
            print(f'  {inj_s}/test={ut_s}: AUROC={auc:.4f}  '
                  f'(n={test_m.sum()}, succ={int(y_te.sum())}, fail={int((1-y_te).sum())})')
            xut_results.append({'injection': inj_s, 'test_ut': ut_s,
                                 'auroc': round(auc,4), 'n': int(test_m.sum())})

    if xut_results:
        xut_df = pd.DataFrame(xut_results)
        print('\nSummary:')
        print(xut_df.to_string(index=False))
        print()
        print('Interpretation:')
        print('  AUROC >> 0.5 → signature transfers across user tasks (environment structure)')
        print('  AUROC ~ 0.5  → signature is user-task-specific, not generalisable')
        xut_df.to_csv(LOG_DIR / 'xut_results.csv', index=False)


---
## Part 2 — Contrastive causal analysis: what changes when outcome flips?

Group runs by `(injection_task, user_task, true_N)`. Within each group everything
is controlled — same payload, same context, same N. Only which tokens were flipped
and the outcome differ. The attention delta is a cleaner causal signal than the
global classifier because context confounds are removed within groups.

Two signals:
- **Δspan**: attention mass shift per (layer, head, semantic span group)
- **Δflip**: attention shift specifically to the substituted token positions

Aggregated by sign consistency × magnitude across all groups.


In [ ]:
def _span_mask(spans, seq_len, tag_set):
    m = torch.zeros(seq_len, dtype=torch.bool)
    for s in spans:
        if s.get('tag') in tag_set:
            m[s['start']:min(s['end'], seq_len)] = True
    for p in SINK_POSITIONS:
        if p < seq_len:
            m[p] = False
    return m


def extract_run_features(cap_dict, ci, flipped_positions):
    '''
    Returns:
      span_mass : [N_LAYERS, N_HEADS, N_GROUPS]  L1-normalised, step 0
      flip_traj : [N_LAYERS, N_HEADS, N_STEPS]   raw mass to flipped tokens, all steps
    '''
    attn    = cap_dict.get('attention', {})
    spans   = ci.get('spans', [])
    seq_len = ci.get('seq_len', 0)

    group_idxs = {}
    for grp, tag_set in SPAN_GROUPS.items():
        mask = _span_mask(spans, seq_len, tag_set)
        group_idxs[grp] = mask.nonzero(as_tuple=False).view(-1)
    sink_idx = torch.tensor([p for p in SINK_POSITIONS if p < seq_len], dtype=torch.long)
    group_idxs['sink'] = sink_idx

    flip_idx = torch.tensor([p for p in flipped_positions if p < seq_len], dtype=torch.long)

    span_mass = np.zeros((N_LAYERS, N_HEADS, N_GROUPS), dtype=np.float32)
    flip_traj = np.zeros((N_LAYERS, N_HEADS, N_REASONING_STEPS), dtype=np.float32)

    for li in range(N_LAYERS):
        if li not in attn:
            continue
        a            = attn[li].float()                    # [H, actual_steps, klen]
        klen         = a.shape[-1]
        actual_steps = min(N_REASONING_STEPS, a.shape[1])

        a0  = a[:, 0, :]                                   # [H, klen]
        tot = a0.sum(-1, keepdim=True).clamp(min=1e-10)
        a0n = a0 / tot

        for gi, grp in enumerate(GROUP_NAMES):
            idxs = group_idxs[grp]
            idxs = idxs[idxs < klen]
            if len(idxs) > 0:
                span_mass[li, :, gi] = a0n[:, idxs].sum(-1).numpy()

        flip_valid = flip_idx[flip_idx < klen]
        if len(flip_valid) > 0:
            for s in range(actual_steps):
                flip_traj[li, :, s] = a[:, s, flip_valid].sum(-1).numpy()

    return span_mass, flip_traj


def compute_group_delta(group_key, group_info):
    inj, ut, N = group_key
    ctx = ('banking', inj, ut)

    success_span, success_flip = [], []
    failure_span, failure_flip = [], []

    for rid in group_info['run_ids']:
        m    = meta_by_rid[rid]
        ci   = cap_by_run.get(rid, {})
        path = TENSOR_BASE / ci.get('cap_path', '')
        if not path.exists():
            continue
        try:
            cap = torch.load(path, map_location='cpu', weights_only=False)
        except Exception:
            continue
        try:
            sp, ft = extract_run_features(cap, ci, m.get('flipped_positions', []))
        except Exception:
            del cap
            continue
        del cap

        if m.get('success'):
            success_span.append(sp)
            success_flip.append(ft)
        else:
            failure_span.append(sp)
            failure_flip.append(ft)

    if not success_span or not failure_span:
        return None

    ms = np.mean(success_span, axis=0)
    mf = np.mean(failure_span, axis=0)
    fs = np.mean(success_flip, axis=0)
    ff = np.mean(failure_flip, axis=0)

    return {
        'delta_span': ms - mf,
        'delta_flip': fs - ff,
        'mean_span_success': ms,
        'mean_span_failure': mf,
        'mean_flip_success': fs,
        'mean_flip_failure': ff,
        'n_success': len(success_span),
        'n_failure': len(failure_span),
        'has_flip_positions': N > 0 and bool(meta_by_rid[group_info['run_ids'][0]].get('flipped_positions')),
    }

print('Feature extraction helpers defined.')


In [ ]:
FORCE_RECOMPUTE = False

if CACHE_FILE.exists() and not FORCE_RECOMPUTE:
    with open(CACHE_FILE, 'rb') as f:
        group_deltas = pickle.load(f)
    print(f'Loaded from cache: {len(group_deltas)} groups')
else:
    def _compute_one(args):
        key, info = args
        return key, compute_group_delta(key, info)

    group_deltas = {}
    with ThreadPoolExecutor(max_workers=6) as ex:
        futs = {ex.submit(_compute_one, (k, v)): k for k, v in mixed_groups.items()}
        for fut in tqdm(as_completed(futs), total=len(futs), desc='Group deltas'):
            key, result = fut.result()
            if result is not None:
                group_deltas[key] = result

    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(group_deltas, f)
    print(f'Computed {len(group_deltas)} group deltas → saved to {CACHE_FILE}')

total_s = sum(d['n_success'] for d in group_deltas.values())
total_f = sum(d['n_failure'] for d in group_deltas.values())
print(f'Groups with data: {len(group_deltas)}')
print(f'Total runs in analysis: {total_s + total_f}  ({total_s} succ / {total_f} fail)')


In [ ]:
all_deltas_span = np.stack([d['delta_span'] for d in group_deltas.values()], axis=0)
# shape: [n_groups, N_LAYERS, N_HEADS, N_GROUPS]

n_groups        = all_deltas_span.shape[0]
mean_delta_span = all_deltas_span.mean(axis=0)         # [L, H, G]
sign_frac       = (all_deltas_span > 0).mean(axis=0)   # fraction where success > failure
sign_strength   = np.abs(sign_frac - 0.5) * 2          # 0=random, 1=perfect
causal_score    = np.abs(mean_delta_span) * sign_strength  # [L, H, G]

print(f'Groups analysed: {n_groups}')
print(f'mean_delta range: [{mean_delta_span.min():.4f}, {mean_delta_span.max():.4f}]')
print()

# ── Print span-group summary (global layers) ──────────────────────────────────
print('Span-group delta  (global layers, mean across all L,H):')
print(f'  {"Group":22s}  {"Δ (succ−fail)":>14}  {"sign_strength":>14}  direction')
print('  ' + '-'*65)
global_delta = mean_delta_span[GLOBAL_LAYERS].mean(axis=(0, 1))   # [G]
global_str   = sign_strength[GLOBAL_LAYERS].mean(axis=(0, 1))
order = np.argsort(np.abs(global_delta))[::-1]
for gi in order:
    grp = GROUP_NAMES[gi]
    d   = global_delta[gi]
    s   = global_str[gi]
    direction = 'MORE in success' if d > 0 else 'LESS in success'
    print(f'  {grp:22s}  {d:+14.5f}  {s:14.3f}  {direction}')


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 10))

# 1. Mean Δ by span group (all layers)
ax = axes[0, 0]
grp_delta = mean_delta_span.mean(axis=(0, 1))
colors = ['#c0392b' if v > 0 else '#2980b9' for v in grp_delta]
bars = ax.barh(GROUP_NAMES, grp_delta, color=colors, edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Mean Δ attention mass (success − failure)')
ax.set_title('Span-group mean delta\n(averaged over all L, H, groups)')

# 2. Causal score by span group
ax = axes[0, 1]
grp_score = causal_score.sum(axis=(0, 1))
ax.barh(GROUP_NAMES, grp_score,
        color=[GROUP_COLOR.get(g, '#888') for g in GROUP_NAMES],
        edgecolor='white', alpha=0.85)
ax.set_xlabel('Causal score  |Δ| × sign_strength  (summed over L, H)')
ax.set_title('Span-group causal score')

# 3. Layer × head heatmap: causal score summed over span groups
ax = axes[0, 2]
lh_score = causal_score.sum(axis=-1)   # [L, H]
im = ax.imshow(lh_score.T, aspect='auto', cmap='hot', interpolation='nearest', origin='lower')
ax.set_xlabel('Layer'); ax.set_ylabel('Head')
ax.set_title('Causal score per (layer, head)\nL=local  G=global')
ax.set_xticks(range(N_LAYERS))
ax.set_xticklabels([f'{l}\n{"L" if l in LOCAL_LAYERS else "G"}' for l in range(N_LAYERS)], fontsize=5)
plt.colorbar(im, ax=ax, shrink=0.8)

# 4. Sign strength heatmap
ax = axes[1, 0]
lh_sign = sign_strength.sum(axis=-1)
im = ax.imshow(lh_sign.T, aspect='auto', cmap='RdYlGn', interpolation='nearest', origin='lower', vmin=0)
ax.set_xlabel('Layer'); ax.set_ylabel('Head')
ax.set_title('Sign consistency per (layer, head)\n(summed over groups, 0=random)')
ax.set_xticks(range(N_LAYERS))
ax.set_xticklabels([f'{l}\n{"L" if l in LOCAL_LAYERS else "G"}' for l in range(N_LAYERS)], fontsize=5)
plt.colorbar(im, ax=ax, shrink=0.8)

# 5. Global layers: Δ attack_payload
ax = axes[1, 1]
payload_gi = GROUP_NAMES.index('attack_payload')
g_payload  = mean_delta_span[GLOBAL_LAYERS, :, payload_gi]
vmax = np.abs(g_payload).max()
im = ax.imshow(g_payload.T, aspect='auto', cmap='RdBu_r', interpolation='nearest',
               origin='lower', vmin=-vmax, vmax=vmax)
ax.set_xlabel('Global layer'); ax.set_ylabel('Head')
ax.set_title('Δ attention → attack_payload\n(global layers, success−failure)')
ax.set_xticks(range(len(GLOBAL_LAYERS)))
ax.set_xticklabels([str(l) for l in GLOBAL_LAYERS], fontsize=6)
plt.colorbar(im, ax=ax, shrink=0.8)

# 6. Global layers: Δ user_instruction
ax = axes[1, 2]
user_gi = GROUP_NAMES.index('user_instruction')
g_user  = mean_delta_span[GLOBAL_LAYERS, :, user_gi]
vmax = np.abs(g_user).max()
im = ax.imshow(g_user.T, aspect='auto', cmap='RdBu_r', interpolation='nearest',
               origin='lower', vmin=-vmax, vmax=vmax)
ax.set_xlabel('Global layer'); ax.set_ylabel('Head')
ax.set_title('Δ attention → user_instruction\n(global layers, success−failure)')
ax.set_xticks(range(len(GLOBAL_LAYERS)))
ax.set_xticklabels([str(l) for l in GLOBAL_LAYERS], fontsize=6)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle(f'Contrastive attention: success − failure  ({n_groups} matched groups)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Only use groups where we have exact flipped positions (not additive fallback)
valid_flip = {k: d for k, d in group_deltas.items()
              if k[2] > 0 and d.get('has_flip_positions')}
print(f'Groups with known flip positions: {len(valid_flip)} / {len(group_deltas)}')

lh_flip_score = None   # set below if we have data

if not valid_flip:
    print('No flip-position data -- all runs used additive-N fallback. Skipping.')
else:
    all_flip = np.stack([d['delta_flip'] for d in valid_flip.values()], axis=0)
    # shape: [n_valid, N_LAYERS, N_HEADS, N_STEPS]

    mean_flip  = all_flip.mean(axis=0)                # [L, H, steps]
    sign_f     = (all_flip > 0).mean(axis=0)
    sign_str_f = np.abs(sign_f - 0.5) * 2
    score_flip = np.abs(mean_flip) * sign_str_f

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    ax = axes[0]
    step0 = mean_flip[:, :, 0]
    vmax  = np.abs(step0).max()
    im = ax.imshow(step0.T, aspect='auto', cmap='RdBu_r', interpolation='nearest',
                   origin='lower', vmin=-vmax, vmax=vmax)
    ax.set_xlabel('Layer'); ax.set_ylabel('Head')
    ax.set_title('Delta attention to flipped tokens (step 0)\nRed = success attends MORE to substituted positions')
    ax.set_xticks(range(N_LAYERS))
    ax.set_xticklabels([f'{l}\n{"L" if l in LOCAL_LAYERS else "G"}' for l in range(N_LAYERS)], fontsize=5)
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax = axes[1]
    im = ax.imshow(score_flip[:, :, 0].T, aspect='auto', cmap='hot',
                   interpolation='nearest', origin='lower')
    ax.set_xlabel('Layer'); ax.set_ylabel('Head')
    ax.set_title('Causal score: attention to flipped tokens (step 0)')
    ax.set_xticks(range(N_LAYERS))
    ax.set_xticklabels([f'{l}\n{"L" if l in LOCAL_LAYERS else "G"}' for l in range(N_LAYERS)], fontsize=5)
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax = axes[2]
    traj_delta = mean_flip.mean(axis=(0, 1))      # [steps]
    traj_score = score_flip.mean(axis=(0, 1))
    steps = range(N_REASONING_STEPS)
    ax.bar(steps, traj_delta, alpha=0.75, color='#3498db', label='Mean delta (success-failure)')
    ax2 = ax.twinx()
    ax2.plot(steps, traj_score, 'o-', color='#e74c3c', lw=2, label='Causal score')
    ax.axhline(0, color='black', lw=0.5)
    ax.set_xlabel('Reasoning step'); ax.set_ylabel('Mean delta attention to flipped positions')
    ax2.set_ylabel('Causal score')
    ax.set_title('Trajectory: attention to flipped tokens\nover first 10 reasoning steps')
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2)

    plt.suptitle(f'Attention to specifically-substituted token positions  ({len(valid_flip)} groups)',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

    lh_flip_score = score_flip[:, :, 0]   # [L, H]


In [ ]:
lh_span_score = causal_score.sum(axis=-1)   # [L, H]

if lh_flip_score is not None:
    span_norm = lh_span_score / (lh_span_score.max() + 1e-10)
    flip_norm = lh_flip_score / (lh_flip_score.max() + 1e-10)
    combined  = (span_norm + flip_norm) / 2
else:
    combined  = lh_span_score / (lh_span_score.max() + 1e-10)

flat      = combined.reshape(-1)
TOP_K     = 30
top_idx   = np.argsort(flat)[::-1][:TOP_K]

rows = []
for rank, idx in enumerate(top_idx, 1):
    li  = idx // N_HEADS
    hd  = idx % N_HEADS
    lg  = 'L' if li in LOCAL_LAYERS else 'G'
    # dominant span group for this head
    dom_gi  = np.argmax(causal_score[li, hd, :])
    dom_grp = GROUP_NAMES[dom_gi]
    dom_dir = 'MORE' if mean_delta_span[li, hd, dom_gi] > 0 else 'LESS'
    rows.append({
        'rank': rank, 'layer': li, 'head': hd, 'L/G': lg,
        'combined': round(float(flat[idx]), 4),
        'span_score': round(float(lh_span_score[li, hd]), 6),
        'flip_score': round(float(lh_flip_score[li, hd]), 6) if lh_flip_score is not None else float('nan'),
        'dominant_group': dom_grp,
        'direction': dom_dir,
    })

top_df = pd.DataFrame(rows)
print(f'Top {TOP_K} causal (layer, head) pairs:')
print(top_df.to_string(index=False))

# ── Shared-importance heatmap (min across per-injection averages) ──────────────
# For each injection, average the combined score across its groups.
# Then take the element-wise min across injections.
inj_scores = {}
for (inj, ut, N), d in group_deltas.items():
    sc = np.abs(d['delta_span']).sum(axis=-1)   # [L, H]
    if inj not in inj_scores:
        inj_scores[inj] = []
    inj_scores[inj].append(sc)

if len(inj_scores) > 1:
    norm_inj = []
    for inj, scs in inj_scores.items():
        mean_sc = np.mean(scs, axis=0)
        norm_inj.append(mean_sc / (mean_sc.max() + 1e-10))
    shared = np.stack(norm_inj).min(axis=0)   # [L, H]

    fig, ax = plt.subplots(figsize=(16, 4))
    im = ax.imshow(shared.T, aspect='auto', cmap='hot', interpolation='nearest', origin='lower')
    ax.set_xlabel('Layer'); ax.set_ylabel('Head')
    ax.set_title('Shared importance across injections (element-wise min of normalised scores)\nBright = consistently discriminative regardless of which injection is being studied')
    ax.set_xticks(range(N_LAYERS))
    ax.set_xticklabels([f'{l}\n{"L" if l in LOCAL_LAYERS else "G"}' for l in range(N_LAYERS)], fontsize=5)
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()


In [ ]:
# Does a classifier using ONLY the top-K head features achieve good AUROC?
# Group-LOO: train on all groups except one, test on the held-out group.
# This validates that the ranked heads are genuinely discriminative.

TOP_K_VAL = 15
top_head_list = [(int(row['layer']), int(row['head'])) for _, row in top_df.head(TOP_K_VAL).iterrows()]
print(f'Building validation matrix with top-{TOP_K_VAL} heads...')

feat_rows, labels, group_keys_list = [], [], []

for key, d in group_deltas.items():
    inj, ut, N = key
    for rid in mixed_groups[key]['run_ids']:
        m    = meta_by_rid[rid]
        ci   = cap_by_run.get(rid, {})
        path = TENSOR_BASE / ci.get('cap_path', '')
        if not path.exists():
            continue
        try:
            cap = torch.load(path, map_location='cpu', weights_only=False)
        except Exception:
            continue

        attn    = cap.get('attention', {})
        spans   = ci.get('spans', [])
        seq_len = ci.get('seq_len', 0)

        row_feat = []
        for li, hd in top_head_list:
            if li not in attn:
                row_feat.extend([0.0] * N_GROUPS)
                continue
            a    = attn[li].float()
            klen = a.shape[-1]
            a0n  = a[hd, 0, :klen]
            a0n  = a0n / (a0n.sum() + 1e-10)
            for grp in GROUP_NAMES:
                if grp == 'sink':
                    idxs = torch.tensor([p for p in SINK_POSITIONS if p < klen])
                else:
                    mask = _span_mask(spans, seq_len, SPAN_GROUPS[grp])
                    idxs = mask.nonzero(as_tuple=False).view(-1)
                    idxs = idxs[idxs < klen]
                row_feat.append(float(a0n[idxs].sum()) if len(idxs) > 0 else 0.0)
        del cap

        feat_rows.append(row_feat)
        labels.append(int(bool(m.get('success'))))
        group_keys_list.append(key)

X_val = np.array(feat_rows, dtype=np.float32)
y_val = np.array(labels, dtype=np.int32)
print(f'Validation matrix: {X_val.shape}   y: {y_val.sum()} succ / {(1-y_val).sum()} fail')

# Group-LOO: leave one (inj, ut, N) group out at a time
unique_gkeys = list(set(group_keys_list))
aucs = []
for test_key in tqdm(unique_gkeys, desc='Group-LOO'):
    test_mask  = np.array([k == test_key for k in group_keys_list])
    train_mask = ~test_mask
    if y_val[test_mask].sum() < 2 or (1 - y_val[test_mask]).sum() < 2:
        continue
    if train_mask.sum() < 30:
        continue
    clf = LogisticRegression(C=0.1, class_weight='balanced', max_iter=500, solver='lbfgs')
    clf.fit(X_val[train_mask], y_val[train_mask])
    y_prob = clf.predict_proba(X_val[test_mask])[:, 1]
    aucs.append(roc_auc_score(y_val[test_mask], y_prob))

print(f'\nGroup-LOO AUROC  (top-{TOP_K_VAL} heads, LogReg):')
print(f'  n groups evaluated : {len(aucs)}')
print(f'  mean  : {np.mean(aucs):.4f}')
print(f'  median: {np.median(aucs):.4f}')
print(f'  min   : {np.min(aucs):.4f}')
print(f'  max   : {np.max(aucs):.4f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(aucs, bins=25, color='#3498db', edgecolor='white', alpha=0.85)
ax.axvline(np.mean(aucs), color='#e74c3c', lw=2, label=f'Mean = {np.mean(aucs):.3f}')
ax.axvline(0.5, color='grey', lw=1, ls='--', label='Chance')
ax.set_xlabel('AUROC'); ax.set_ylabel('Count')
ax.set_title(f'Group-LOO AUROC using only top-{TOP_K_VAL} causal heads  (n={len(aucs)} groups)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
print('TOKEN SUBSTITUTION GUIDANCE')
print('=' * 70)
print()
print('What does a successful run look like, attention-wise?')
print()

global_delta = mean_delta_span[GLOBAL_LAYERS].mean(axis=(0, 1))   # [G]
global_str   = sign_strength[GLOBAL_LAYERS].mean(axis=(0, 1))
order        = np.argsort(np.abs(global_delta * global_str))[::-1]

print(f'  {"Group":22s}  {"delta (succ-fail)":>18}  {"consistency":>12}  interpretation')
print('  ' + '-'*75)
for gi in order[:10]:
    grp = GROUP_NAMES[gi]
    d   = global_delta[gi]
    s   = global_str[gi]
    if abs(d) < 1e-5:
        continue
    direction = 'MORE in success' if d > 0 else 'LESS in success'
    print(f'  {grp:22s}  {d:+18.5f}  {s:12.3f}  {direction}')

print()
if lh_flip_score is not None:
    flip_delta_mean = np.stack([d['delta_flip'][:,:,0] for d in valid_flip.values()]).mean(0)
    pos_flip = (flip_delta_mean[GLOBAL_LAYERS] > 0).mean()
    print(f'Global heads attending MORE to flipped positions in success: {pos_flip:.1%}')
    if pos_flip > 0.6:
        print('Global head flip strategy: NOTICE  (success attends more to substituted positions)')
    elif pos_flip < 0.4:
        print('Global head flip strategy: IGNORE  (success attends less to substituted positions)')
    else:
        print('Global head flip strategy: MIXED   (no consistent direction)')
    print()

print('ACTIONABLE STRATEGIES:')
print('-' * 70)

payload_delta = global_delta[GROUP_NAMES.index('attack_payload')]
user_delta    = global_delta[GROUP_NAMES.index('user_instruction')]

if payload_delta > 0:
    print('Strategy A - Payload salience:')
    print('  Success has MORE global attention on the attack_payload span.')
    print('  Substitute tokens that increase L1-mass from global heads to payload.')
    print('  Surrogate objective: score(t) = sum_{(l,h) in top_heads} attn[l,h,0,payload_positions]')
    print('  Prefer replacements that are syntactically prominent (nouns, topic words).')
else:
    print('Strategy A - Payload camouflage:')
    print('  Success has LESS global attention on the attack_payload span.')
    print('  The injection works by blending in, not standing out.')
    print('  Substitute with contextually coherent tokens that do not attract attention.')

if user_delta < 0:
    print()
    print('Strategy B - User distraction:')
    print('  Success has LESS global attention on user_instruction.')
    print('  Substitutions that make user instruction harder to parse may help.')

print()
print('Strategy C - Causal-head-guided discrete search:')
print('  Use top-K causal heads as a surrogate reward for token substitution:')
print('    for each position p in payload (excluding vital strings):')
print('      for each candidate token t:')
print('        forward-pass with t at p')
print('        reward = sum_{(l,h) in top_causal_heads} delta_weight[l,h] * attn[l,h,0,p]')
print('      select t* = argmax(reward)')
print('  No output logit gradient needed -- uses only internal attention values.')
print('  Vital strings (IBANs, emails, phone numbers) must be excluded.')

print()
print('Top causal heads to use as surrogate objective:')
print(top_df[['rank','layer','head','L/G','combined','dominant_group','direction']].head(10).to_string(index=False))


## Position-level causal analysis (N=1 only)

For runs with exactly one token flipped we know the *position* of the flip from
`perturbation_position_start` in the run metadata.

Group by `(injection_task, user_task, position_p)`. Within each group every run
modified the same payload token slot — the only variation is which replacement token
was chosen and the outcome. This eliminates token-identity as a confound for the
attention delta.

Three outcome classes:
- **Mixed**: position p, same (inj, ut), some flips succeed and some fail.
  Delta here is purely context-driven — the model is behaving differently not because
  of the token but because of something about the interaction between that position
  and the current user-task framing.
- **Always-success**: flipping p never breaks the injection in any tested context →
  structurally unimportant position.
- **Always-failure**: flipping p always breaks the injection → structurally critical,
  injection cannot survive modification there.


---
## Part 3 — Position-level causal analysis: which payload tokens are critical?

For N=1 runs where a single token was flipped, group by
`(injection_task, user_task, payload_position)`. The replacement token varies within
each group — token identity is controlled at the position level.

Three outcome classes reveal the structural importance of each position:
- **Always-success**: injection tolerates any flip here — structurally unimportant
- **Always-failure**: injection cannot survive a flip here — structurally critical
- **Mixed**: context-dependent — the same position flip sometimes works and sometimes
  doesn't. The attention delta here eliminates token identity as a confound entirely.


In [ ]:
# ── Build position-level groups ───────────────────────────────────────────────
pos_groups = {}   # (inj, ut, pos) -> {'success': [rids], 'failure': [rids]}

for m in flip_runs:
    if m.get('source') != 'perturbation': continue
    if int(m.get('perturbation_N', 0)) != 1: continue
    pos = m.get('perturbation_position_start')
    if pos is None: continue

    key = (m['injection_task_id'], m['user_task_id'], int(pos))
    if key not in pos_groups:
        pos_groups[key] = {'success': [], 'failure': []}
    bucket = 'success' if m.get('success') else 'failure'
    pos_groups[key][bucket].append(m['run_id'])

mixed_pos    = {k: v for k, v in pos_groups.items() if v['success'] and v['failure']}
always_suc   = {k: v for k, v in pos_groups.items() if v['success'] and not v['failure']}
always_fail  = {k: v for k, v in pos_groups.items() if v['failure'] and not v['success']}

print(f'N=1 position groups: {len(pos_groups)}')
print(f'  Mixed (context-dependent):  {len(mixed_pos):4d}  '
      f'({sum(len(v["success"])+len(v["failure"]) for v in mixed_pos.values())} runs)')
print(f'  Always success:             {len(always_suc):4d}  '
      f'({sum(len(v["success"]) for v in always_suc.values())} runs)')
print(f'  Always failure:             {len(always_fail):4d}  '
      f'({sum(len(v["failure"]) for v in always_fail.values())} runs)')

# ── Payload position importance map ───────────────────────────────────────────
from collections import defaultdict

pos_outcome = defaultdict(lambda: {'mixed':0,'always_suc':0,'always_fail':0,'total':0})
for (inj, ut, pos), v in pos_groups.items():
    d = pos_outcome[(inj.replace('injection_task_','it'), pos)]
    d['total'] += 1
    if v['success'] and v['failure']:  d['mixed']      += 1
    elif v['success']:                 d['always_suc'] += 1
    else:                              d['always_fail'] += 1

# Per-injection position heatmap
inj_list = sorted(set(k[0] for k in pos_outcome.keys()))
n_inj    = len(inj_list)
max_pos  = max(k[1] for k in pos_outcome.keys()) + 1

fig, axes = plt.subplots(n_inj, 1, figsize=(18, 2.5 * n_inj), squeeze=False)
for ai, inj in enumerate(inj_list):
    ax      = axes[ai][0]
    row_m   = np.zeros(max_pos)
    row_suc = np.zeros(max_pos)
    row_fai = np.zeros(max_pos)
    for (inj2, pos), d in pos_outcome.items():
        if inj2 != inj: continue
        row_m[pos]   = d['mixed']
        row_suc[pos] = d['always_suc']
        row_fai[pos] = d['always_fail']

    x = np.arange(max_pos)
    ax.bar(x, row_suc, color='#2ecc71', alpha=0.8, label='always success')
    ax.bar(x, row_fai, bottom=row_suc, color='#e74c3c', alpha=0.8, label='always failure')
    ax.bar(x, row_m,   bottom=row_suc+row_fai, color='#f39c12', alpha=0.9, label='mixed')
    ax.set_title(f'{inj} — payload position importance (N=1 flips)', fontsize=9)
    ax.set_xlabel('Payload token position')
    ax.set_ylabel('# groups')
    if ai == 0:
        ax.legend(fontsize=8, loc='upper right')

plt.suptitle('Position outcome classes across N=1 flips\n'
             'Orange=context-sensitive  Green=structurally unimportant  Red=critical',
             fontsize=11)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'position_importance_N1.png', bbox_inches='tight')
plt.show(); plt.close('all')

# Summary table
print('\nAlways-failure positions (critical — injection cannot survive flip here):')
print(f'  {"Injection":8s}  {"UserTask":8s}  {"Position":>8s}  {"N_runs":>6s}')
for (inj, ut, pos), v in sorted(always_fail.items(), key=lambda x: x[0]):
    print(f'  {inj.replace("injection_task_","it"):8s}  '
          f'{ut.replace("user_task_","ut"):8s}  {pos:8d}  {len(v["failure"]):6d}')


In [ ]:
# ── Attention delta for mixed groups ──────────────────────────────────────────
# For each mixed (inj, ut, pos) group: load attention tensors, compute
# mean(success) - mean(failure). Token identity is fully controlled — same
# position, same context, only the replacement token and outcome differ.

print(f'Loading attention for {len(mixed_pos)} mixed position groups '
      f'({sum(len(v["success"])+len(v["failure"]) for v in mixed_pos.values())} runs)...')

def _extract_span_mass(cap_dict, ci, step=0):
    attn    = cap_dict.get('attention', {})
    spans   = ci.get('spans', [])
    seq_len = ci.get('seq_len', 0)
    group_idxs = {}
    for grp, tag_set in SPAN_GROUPS.items():
        mask = _span_mask(spans, seq_len, tag_set)
        group_idxs[grp] = mask.nonzero(as_tuple=False).view(-1)
    sink_idx = torch.tensor([p for p in SINK_POSITIONS if p < seq_len], dtype=torch.long)
    group_idxs['sink'] = sink_idx

    sm = np.zeros((N_LAYERS, N_HEADS, N_GROUPS), dtype=np.float32)
    for li in range(N_LAYERS):
        if li not in attn: continue
        a    = attn[li].float()
        klen = a.shape[-1]
        if a.shape[1] <= step: continue
        a0   = a[:, step, :]
        a0n  = a0 / (a0.sum(-1, keepdim=True).clamp(min=1e-10))
        for gi, grp in enumerate(GROUP_NAMES):
            idxs = group_idxs[grp]
            idxs = idxs[idxs < klen]
            if len(idxs) > 0:
                sm[li, :, gi] = a0n[:, idxs].sum(-1).numpy()
    return sm


pos_deltas = []   # list of (delta [L,H,G], n_suc, n_fail, group_key)

for key, v in mixed_pos.items():
    suc_spans, fai_spans = [], []
    for rid in v['success'] + v['failure']:
        ci   = cap_by_run.get(rid, {})
        path = TENSOR_BASE / ci.get('cap_path', '')
        if not path.exists(): continue
        try:
            cap = torch.load(path, map_location='cpu', weights_only=False)
        except Exception:
            continue
        sm = _extract_span_mass(cap, ci)
        del cap
        (suc_spans if rid in v['success'] else fai_spans).append(sm)

    if not suc_spans or not fai_spans: continue
    delta = np.mean(suc_spans, axis=0) - np.mean(fai_spans, axis=0)
    pos_deltas.append((delta, len(suc_spans), len(fai_spans), key))

print(f'Groups with data: {len(pos_deltas)}')

if pos_deltas:
    all_d = np.stack([d for d, *_ in pos_deltas], axis=0)   # [G, L, H, N_G]

    mean_d = all_d.mean(axis=0)          # [L, H, N_G]
    sign_s = np.abs((all_d > 0).mean(axis=0) - 0.5) * 2
    score  = np.abs(mean_d) * sign_s

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    ax = axes[0]
    lh = score.sum(axis=-1)
    im = ax.imshow(lh.T, aspect='auto', cmap='hot', interpolation='nearest', origin='lower')
    ax.set_xlabel('Layer'); ax.set_ylabel('Head')
    ax.set_title('Causal score (position-level, mixed groups only)\n'
                 'Token identity fully controlled')
    ax.set_xticks(range(N_LAYERS))
    ax.set_xticklabels([f'{l}\n{"L" if l in LOCAL_LAYERS else "G"}' for l in range(N_LAYERS)], fontsize=5)
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax = axes[1]
    grp_delta = mean_d.mean(axis=(0,1))
    colors = ['#c0392b' if v > 0 else '#2980b9' for v in grp_delta]
    ax.barh(GROUP_NAMES, grp_delta, color=colors, edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Mean delta (success - failure)')
    ax.set_title('Span-group delta\n(position-controlled pairs)')

    ax = axes[2]
    global_payload = mean_d[GLOBAL_LAYERS, :, GROUP_NAMES.index('attack_payload')]
    vmax = np.abs(global_payload).max()
    im2 = ax.imshow(global_payload.T, aspect='auto', cmap='RdBu_r',
                    interpolation='nearest', origin='lower', vmin=-vmax, vmax=vmax)
    ax.set_xlabel('Global layer'); ax.set_ylabel('Head')
    ax.set_title('Delta attention to attack_payload\n(global layers, position-controlled)')
    ax.set_xticks(range(len(GLOBAL_LAYERS)))
    ax.set_xticklabels([str(l) for l in GLOBAL_LAYERS], fontsize=6)
    plt.colorbar(im2, ax=ax, shrink=0.8)

    plt.suptitle(f'Position-level causal delta ({len(pos_deltas)} mixed groups, N=1)\n'
                 'Same position, same context — only replacement token and outcome vary',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'position_causal_delta.png', bbox_inches='tight')
    plt.show(); plt.close('all')

    # Compare to N-level causal score (does position-controlled analysis agree?)
    pos_lh_score = score.sum(axis=-1)
    nle_lh_score = causal_score.sum(axis=-1) if 'causal_score' in dir() else None
    if nle_lh_score is not None:
        pos_flat = pos_lh_score.reshape(-1)
        nle_flat = nle_lh_score.reshape(-1)
        from scipy.stats import spearmanr
        rho, pval = spearmanr(pos_flat, nle_flat)
        print(f'\nSpearman rank correlation vs N-level causal score: rho={rho:.3f}  p={pval:.3e}')
        print('High correlation = position-controlled analysis agrees with N-level grouping')
        print('Low correlation  = token identity was confounding the N-level signal')


---
## Part 4 — Cross-reference: do discriminative heads = causal heads?

The LOI identifies heads that *discriminate* success from failure across unseen
injection and user-task contexts. The contrastive delta analysis (Part 2) identifies
heads whose attention *changes direction* consistently when outcome flips within
matched groups.

If these two rankings agree strongly, the discriminative signal and the causal
mechanism are the same thing — evidence that the attention shift is the proximate
cause of generalisation. If they diverge, the classifier may be latching onto
correlates rather than causes.


In [ ]:
# ── Cross-reference: LOI importance vs contrastive causal score ───────────────
# Both are [n_layers, H] arrays. Compare their rankings via Spearman correlation.

if 'LOI_LH_IMP' not in dir() or 'causal_score' not in dir():
    print('Run the LOI cells and the contrastive delta cells first.')
else:
    from scipy.stats import spearmanr

    layers_loi    = GLOBAL_LAYERS if 'global' in BEST_LOI_FEAT else list(range(N_LAYERS))
    loi_flat      = LOI_LH_IMP.reshape(-1)

    # Causal score over same layers
    caus_lh       = causal_score[layers_loi].sum(axis=-1)   # [n_layers, H]
    caus_flat     = caus_lh.reshape(-1)

    rho, pval = spearmanr(loi_flat, caus_flat)
    print(f'Spearman rank correlation  (LOI importance vs contrastive causal score):')
    print(f'  rho = {rho:.3f}   p = {pval:.3e}')
    print()
    if rho > 0.5:
        print('  Strong agreement: discriminative and causal signals point to the same heads.')
        print('  The attention shift IS the mechanism — not just a correlate.')
    elif rho > 0.2:
        print('  Moderate agreement: partial overlap.')
        print('  Some causal heads are detected by the classifier, others are not.')
    else:
        print('  Weak agreement: discriminative and causal signals diverge.')
        print('  Classifier may be using a correlate; causal analysis gives cleaner signal.')

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    ax = axes[0]
    im = ax.imshow(LOI_LH_IMP.T, aspect='auto', cmap='viridis',
                   interpolation='nearest', origin='lower')
    ax.set_title(f'LOI importance ({BEST_LOI_FEAT})')
    ax.set_xlabel('Layer'); ax.set_ylabel('Head')
    ax.set_xticks(range(len(layers_loi)))
    ax.set_xticklabels([f'{l}\n{"G" if l not in LOCAL_LAYERS else "L"}' for l in layers_loi], fontsize=5)
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax = axes[1]
    im2 = ax.imshow(caus_lh.T, aspect='auto', cmap='hot',
                    interpolation='nearest', origin='lower')
    ax.set_title('Contrastive causal score (matched groups)')
    ax.set_xlabel('Layer'); ax.set_ylabel('Head')
    ax.set_xticks(range(len(layers_loi)))
    ax.set_xticklabels([f'{l}\n{"G" if l not in LOCAL_LAYERS else "L"}' for l in layers_loi], fontsize=5)
    plt.colorbar(im2, ax=ax, shrink=0.8)

    ax = axes[2]
    ax.scatter(loi_flat, caus_flat, alpha=0.3, s=8, color='#2c3e50')
    ax.set_xlabel('LOI importance (discriminative)')
    ax.set_ylabel('Causal score (contrastive)')
    ax.set_title(f'Head-level scatter\nSpearman rho={rho:.3f}  p={pval:.2e}')
    # label top-5 by causal score
    top5 = np.argsort(caus_flat)[::-1][:5]
    for idx in top5:
        li = idx // N_HEADS; hd = idx % N_HEADS
        lg = 'G' if layers_loi[li] not in LOCAL_LAYERS else 'L'
        ax.annotate(f'{lg}{layers_loi[li]}/H{hd}',
                    (loi_flat[idx], caus_flat[idx]), fontsize=7,
                    xytext=(4,4), textcoords='offset points')

    plt.suptitle('LOI discriminative importance vs contrastive causal score',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'loi_vs_causal_crossref.png', bbox_inches='tight')
    plt.show(); plt.close('all')
